# DANI Model Metrics Evaluation Notebook
## Fast, Reliable, and Comprehensive Metric Calculation

This notebook:
- Automatically installs all required dependencies
- Intelligently evaluates snapshots (with skip options for speed)
- Handles the CUDA plugin compilation errors gracefully
- Stores results in clean CSV format for easy comparison
- Provides visualization and statistics

## Cell 1: Install Dependencies
Run this first. It installs all packages into your current Jupyter kernel.

In [1]:
# Install all required packages
%pip install pandas numpy matplotlib scikit-image pillow --quiet
print("✓ All packages installed successfully!")

Note: you may need to restart the kernel to use updated packages.
✓ All packages installed successfully!



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 2: Import Libraries & Set Configuration

In [2]:
import os
import sys
import glob
import subprocess
import json
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful!")

✓ All imports successful!


## Cell 3: Configuration - CUSTOMIZE THIS CELL

In [9]:

import os
import sys
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

# ============================================================================
# CONFIGURATION - EDIT THESE PATHS FOR YOUR SETUP
# ============================================================================

# === CRITICAL PATHS ===
DANI_DIR = os.path.join(BASE_DIR, "src", "DANI")
RUN_DIR = os.path.join(BASE_DIR, "src", "DANI", "training-runs", "00004-stylegan2-panda-gpus1-batch8-d_pos-first-noise_sd-0.5-target0.45-ada_kimg100-brand_new_run_numbered")
DATASET_PATH = os.path.join(BASE_DIR, "src", "FakeCLR", "data", "panda.zip")
PYTHON_EXEC = os.path.join(BASE_DIR, "src", "DANI", "dani_env", "Scripts", "python.exe")

# === METRICS ===
# Available metrics (check your DANI/metrics/metric_main.py for full list):
# Common ones: fid50k_full, is50k, kid50k_full, lpips
# For low-shot (100 images): kid50k_full is more stable than fid50k_full
METRICS = "fid50k_full"

# === EVALUATION SPEED ===
# Skip every N snapshots. 1=evaluate all, 5=evaluate every 5th, 10=every 10th
# For 100+ snapshots on a limited dataset, use 5 or 10 to save days of computation
STEP_SIZE = 1

# GPU to use (if you have multiple GPUs)
GPU_ID = "0"

# === OUTPUT ===
# Where to save the final CSV
OUTPUT_CSV = os.path.join(RUN_DIR, "compiled_metrics_over_time.csv")
BACKUP_DIR = os.path.join(RUN_DIR, "metrics_backup")

# ============================================================================

# Verify paths exist
paths_to_check = {
    "DANI_DIR": DANI_DIR,
    "RUN_DIR": RUN_DIR,
    "DATASET_PATH": DATASET_PATH,
    "PYTHON_EXEC": PYTHON_EXEC
}

print("\n" + "="*70)
print("CONFIGURATION VERIFICATION")
print("="*70)

all_valid = True
for name, path in paths_to_check.items():
    exists = os.path.exists(path)
    status = "✓" if exists else "✗"
    print(f"{status} {name:20s}: {path}")
    if not exists:
        all_valid = False

print("\nEvaluation Settings:")
print(f"  Metrics: {METRICS}")
print(f"  Step Size (skip every N): {STEP_SIZE}")
print(f"  GPU ID: {GPU_ID}")
print(f"  Output CSV: {OUTPUT_CSV}")
print("="*70 + "\n")

if not all_valid:
    print("\n⚠️  ERROR: Some paths are missing! Please update the configuration above.")
    print("Double-check your folder names and paths.")
else:
    print("✓ All paths verified successfully!")


CONFIGURATION VERIFICATION
✓ DANI_DIR            : C:\Users\vishw\OneDrive\Desktop\Explo\DANI
✓ RUN_DIR             : C:\Users\vishw\OneDrive\Desktop\Explo\DANI\training-runs\00004-stylegan2-panda-gpus1-batch8-d_pos-first-noise_sd-0.5-target0.45-ada_kimg100-brand_new_run_numbered
✓ DATASET_PATH        : C:\Users\vishw\OneDrive\Desktop\Explo\FakeCLR\data\panda.zip
✓ PYTHON_EXEC         : C:\Users\vishw\OneDrive\Desktop\Explo\DANI\dani_env\Scripts\python.exe

Evaluation Settings:
  Metrics: fid50k_full
  Step Size (skip every N): 1
  GPU ID: 0
  Output CSV: C:\Users\vishw\OneDrive\Desktop\Explo\DANI\training-runs\00004-stylegan2-panda-gpus1-batch8-d_pos-first-noise_sd-0.5-target0.45-ada_kimg100-brand_new_run_numbered\compiled_metrics_over_time.csv

✓ All paths verified successfully!


## Cell 4: Discover Snapshots

In [10]:
# Find all network snapshots
search_pattern = os.path.join(RUN_DIR, "network-snapshot-*.pkl")
all_pkl_files = sorted(glob.glob(search_pattern))

print("\n" + "="*70)
print("SNAPSHOT DISCOVERY")
print("="*70)

if len(all_pkl_files) == 0:
    print(f"\n✗ ERROR: No .pkl files found in {RUN_DIR}")
    print("\nSearching pattern used: " + search_pattern)
    print("\nPlease verify:")
    print(f"  1. RUN_DIR exists: {os.path.exists(RUN_DIR)}")
    print(f"  2. Directory contents:")
    if os.path.exists(RUN_DIR):
        print("\n".join([f"     - {f}" for f in os.listdir(RUN_DIR)[:20]]))
else:
    print(f"\n✓ Found {len(all_pkl_files)} total snapshots")
    print(f"\nFirst snapshot:  {os.path.basename(all_pkl_files[0])}")
    print(f"Last snapshot:   {os.path.basename(all_pkl_files[-1])}")
    
    # Calculate which ones will be evaluated
    pkl_files_to_test = all_pkl_files[::STEP_SIZE]
    
    # Always include the last one
    if len(all_pkl_files) > 0 and all_pkl_files[-1] not in pkl_files_to_test:
        pkl_files_to_test.append(all_pkl_files[-1])
    
    pkl_files_to_test = sorted(pkl_files_to_test)
    
    print(f"\n📊 Evaluation Plan:")
    print(f"   - Total snapshots: {len(all_pkl_files)}")
    print(f"   - Will evaluate: {len(pkl_files_to_test)} snapshots")
    print(f"   - Time saved: ~{100 - (len(pkl_files_to_test)/len(all_pkl_files)*100):.0f}%")
    print(f"\n   Snapshots to evaluate:")
    for i, pkl in enumerate(pkl_files_to_test[:10]):
        print(f"     {i+1}. {os.path.basename(pkl)}")
    if len(pkl_files_to_test) > 10:
        print(f"     ... ({len(pkl_files_to_test)-10} more) ...")
        print(f"     {len(pkl_files_to_test)}. {os.path.basename(pkl_files_to_test[-1])}")

print("\n" + "="*70)


SNAPSHOT DISCOVERY

✓ Found 13 total snapshots

First snapshot:  network-snapshot-000000.pkl
Last snapshot:   network-snapshot-000012.pkl

📊 Evaluation Plan:
   - Total snapshots: 13
   - Will evaluate: 13 snapshots
   - Time saved: ~0%

   Snapshots to evaluate:
     1. network-snapshot-000000.pkl
     2. network-snapshot-000001.pkl
     3. network-snapshot-000002.pkl
     4. network-snapshot-000003.pkl
     5. network-snapshot-000004.pkl
     6. network-snapshot-000005.pkl
     7. network-snapshot-000006.pkl
     8. network-snapshot-000007.pkl
     9. network-snapshot-000008.pkl
     10. network-snapshot-000009.pkl
     ... (3 more) ...
     13. network-snapshot-000012.pkl



## Cell 5: The Main Evaluation Engine

⚠️ **This cell will take a LONG time to run** (hours to days depending on dataset and metric).

It handles:
- CUDA plugin errors gracefully
- Subprocess management
- Real-time progress output
- Error recovery

In [12]:
# Create backup directory
os.makedirs(BACKUP_DIR, exist_ok=True)

# Setup environment to avoid CUDA compilation errors
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = str(GPU_ID) # Make sure this is a string, like "0"
env["PYTORCH_NVFUSER_DISABLE"] = "1"

print("\n" + "="*70)
print("STARTING METRIC EVALUATION")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"GPU: {GPU_ID}")
print(f"Metrics: {METRICS}")
print(f"Total snapshots to evaluate: {len(pkl_files_to_test)}")
print("\nNote: This may take several hours. Progress will be shown below.\n")

evaluation_results = []
successful_evals = 0
failed_evals = 0

for idx, pkl_file in enumerate(pkl_files_to_test, 1):
    snapshot_name = os.path.basename(pkl_file)
    snapshot_num = snapshot_name.replace('network-snapshot-', '').replace('.pkl', '')
    
    print(f"\n[{idx}/{len(pkl_files_to_test)}] Evaluating: {snapshot_name}")
    print("-" * 70)
    
    # Build command
    cmd = [
        PYTHON_EXEC,
        "calc_metrics.py",
        f"--metrics={METRICS}",
        f"--network={pkl_file}",
        f"--data={DATASET_PATH}"
    ]
    
    try:
        # Run subprocess with streaming output
        process = subprocess.Popen(
            cmd,
            cwd=DANI_DIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        
        # Stream output in real-time
        output_lines = []
        for line in process.stdout:
            output_lines.append(line)
            # Only print key lines to reduce clutter
            if any(keyword in line.lower() for keyword in ['fid', 'is50k', 'kid', 'error', 'success', 'metric', 'snapshot']):
                print(line.rstrip())
        
        return_code = process.wait()
        
        if return_code == 0:
            print(f"✓ Successfully evaluated {snapshot_name}")
            successful_evals += 1
            
            # Record result
            evaluation_results.append({
                'snapshot': snapshot_num,
                'pkl_file': snapshot_name,
                'status': 'success',
                'timestamp': datetime.now().isoformat()
            })
        else:
            print(f"✗ Failed to evaluate {snapshot_name} (return code: {return_code})")
            failed_evals += 1
            
            evaluation_results.append({
                'snapshot': snapshot_num,
                'pkl_file': snapshot_name,
                'status': 'failed',
                'timestamp': datetime.now().isoformat()
            })
    
    except Exception as e:
        print(f"✗ Exception while evaluating {snapshot_name}: {str(e)}")
        failed_evals += 1
        evaluation_results.append({
            'snapshot': snapshot_num,
            'pkl_file': snapshot_name,
            'status': 'error',
            'error_msg': str(e),
            'timestamp': datetime.now().isoformat()
        })

print("\n" + "="*70)
print("EVALUATION COMPLETE")
print("="*70)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Successful: {successful_evals}/{len(pkl_files_to_test)}")
print(f"Failed: {failed_evals}/{len(pkl_files_to_test)}")
print("="*70)


STARTING METRIC EVALUATION
Start time: 2026-04-02 23:09:21
GPU: 0
Metrics: fid50k_full
Total snapshots to evaluate: 13

Note: This may take several hours. Progress will be shown below.


[1/13] Evaluating: network-snapshot-000000.pkl
----------------------------------------------------------------------
Loading network from "C:\Users\vishw\OneDrive\Desktop\Explo\DANI\training-runs\00004-stylegan2-panda-gpus1-batch8-d_pos-first-noise_sd-0.5-target0.45-ada_kimg100-brand_new_run_numbered\network-snapshot-000000.pkl"...
  File "C:\Users\vishw\OneDrive\Desktop\Explo\DANI\calc_metrics.py", line 186, in <module>
    calc_metrics() # pylint: disable=no-value-for-parameter
  File "C:\Users\vishw\OneDrive\Desktop\Explo\DANI\calc_metrics.py", line 179, in calc_metrics
  File "C:\Users\vishw\OneDrive\Desktop\Explo\DANI\calc_metrics.py", line 59, in subprocess_fn
    raise RuntimeError(f'Could not find MSVC/GCC/CLANG installation on this computer. Check _find_compiler_bindir() in "{__file__}".')
R

## Cell 6: Parse Metric Results from JSONL Files

In [6]:
# DANI saves results in metric-*.jsonl files. Let's parse them.

print("\n" + "="*70)
print("PARSING METRIC RESULTS")
print("="*70)

# Find all metric JSONL files in the run directory
jsonl_pattern = os.path.join(RUN_DIR, "metric-*.jsonl")
jsonl_files = sorted(glob.glob(jsonl_pattern))

print(f"\nFound {len(jsonl_files)} metric files.")

all_metrics = []
parse_errors = 0

for jsonl_file in jsonl_files:
    print(f"\nParsing: {os.path.basename(jsonl_file)}")
    
    try:
        with open(jsonl_file, 'r') as f:
            for line_num, line in enumerate(f, 1):
                if line.strip():
                    try:
                        data = json.loads(line)
                        all_metrics.append(data)
                    except json.JSONDecodeError as e:
                        print(f"  ⚠️  JSON parse error on line {line_num}: {str(e)[:50]}")
                        parse_errors += 1
                        continue
        
        print(f"  ✓ Successfully parsed {len(all_metrics)} metrics")
    
    except Exception as e:
        print(f"  ✗ Error reading file: {str(e)}")

print(f"\nTotal metrics parsed: {len(all_metrics)}")
if parse_errors > 0:
    print(f"Parse errors encountered: {parse_errors}")

if len(all_metrics) > 0:
    print(f"\nSample metric record:")
    print(json.dumps(all_metrics[0], indent=2))
else:
    print("\n⚠️  No metrics were parsed. This could mean:")
    print("   1. Evaluation failed (check Cell 5 output)")
    print("   2. JSONL files don't exist yet")
    print("   3. Wrong RUN_DIR path")


PARSING METRIC RESULTS

Found 3 metric files.

Parsing: metric-fid50k_full.jsonl
  ✓ Successfully parsed 1 metrics

Parsing: metric-is50k.jsonl
  ✓ Successfully parsed 2 metrics

Parsing: metric-kid50k_full.jsonl
  ✓ Successfully parsed 3 metrics

Total metrics parsed: 3

Sample metric record:
{
  "results": {
    "fid50k_full": 329.38038634031886
  },
  "metric": "fid50k_full",
  "total_time": 683.5611615180969,
  "total_time_str": "11m 24s",
  "num_gpus": 1,
  "snapshot_pkl": "network-snapshot-000000.pkl",
  "timestamp": 1775141247.6793835
}


## Cell 7: Convert to CSV and Save

In [7]:
print("\n" + "="*70)
print("CREATING FINAL CSV")
print("="*70)

if len(all_metrics) > 0:
    # Convert to DataFrame
    df = pd.DataFrame(all_metrics)
    
    print(f"\nDataFrame shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    
    # Sort by snapshot_kimg if available, otherwise by snapshot
    if 'snapshot_kimg' in df.columns:
        df = df.sort_values(by='snapshot_kimg').reset_index(drop=True)
        print(f"\nSorted by snapshot_kimg (training progress)")
    elif 'snapshot' in df.columns:
        df = df.sort_values(by='snapshot').reset_index(drop=True)
        print(f"\nSorted by snapshot number")
    
    # Save to CSV
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✓ CSV saved to:")
    print(f"  {OUTPUT_CSV}")
    
    # Also save as Excel for convenience
    excel_path = OUTPUT_CSV.replace('.csv', '.xlsx')
    try:
        df.to_excel(excel_path, index=False, engine='openpyxl')
        print(f"\n✓ Excel file also saved to:")
        print(f"  {excel_path}")
    except:
        print("\n(openpyxl not installed - skipping Excel export)")
    
    # Save as JSON backup
    json_path = OUTPUT_CSV.replace('.csv', '.json')
    with open(json_path, 'w') as f:
        json.dump(all_metrics, f, indent=2)
    print(f"\n✓ JSON backup saved to:")
    print(f"  {json_path}")
    
    print("\n" + "="*70)
    print("DATA PREVIEW")
    print("="*70)
    print(f"\nFirst few rows:")
    display(df.head(10))
    
    print(f"\nBasic statistics:")
    display(df.describe())
    
else:
    print("\n✗ No metrics to save. Please check the evaluation results above.")


CREATING FINAL CSV

DataFrame shape: (3, 7)
Columns: ['results', 'metric', 'total_time', 'total_time_str', 'num_gpus', 'snapshot_pkl', 'timestamp']

✓ CSV saved to:
  C:\Users\vishw\OneDrive\Desktop\Explo\DANI\training-runs\00004-stylegan2-panda-gpus1-batch8-d_pos-first-noise_sd-0.5-target0.45-ada_kimg100-brand_new_run_numbered\compiled_metrics_over_time.csv

(openpyxl not installed - skipping Excel export)

✓ JSON backup saved to:
  C:\Users\vishw\OneDrive\Desktop\Explo\DANI\training-runs\00004-stylegan2-panda-gpus1-batch8-d_pos-first-noise_sd-0.5-target0.45-ada_kimg100-brand_new_run_numbered\compiled_metrics_over_time.json

DATA PREVIEW

First few rows:


,results,metric,total_time,total_time_str,num_gpus,snapshot_pkl,timestamp
0,{'fid50k_full': 329.38038634031886},fid50k_full,683.561162,11m 24s,1,network-snapshot-000000.pkl,1.775141e+09
1,"{'is50k_mean': 5.310664176940918, 'is50k_std':...",is50k,653.043384,10m 53s,1,network-snapshot-000000.pkl,1.775142e+09
2,{'kid50k_full': 0.30937227606773376},kid50k_full,638.488070,10m 38s,1,network-snapshot-000000.pkl,1.775141e+09



Basic statistics:


,total_time,num_gpus,timestamp
count,3.000000,3.0,3.000000e+00
mean,658.364205,1.0,1.775141e+09
std,23.002809,0.0,6.683630e+02
min,638.488070,1.0,1.775141e+09
25%,645.765727,1.0,1.775141e+09
50%,653.043384,1.0,1.775141e+09
75%,668.302273,1.0,1.775142e+09
max,683.561162,1.0,1.775142e+09


## Cell 8: Visualization & Analysis

In [8]:
import matplotlib.pyplot as plt

if len(all_metrics) > 0:
    print("\n" + "="*70)
    print("GENERATING PLOTS")
    print("="*70)
    
    df = pd.read_csv(OUTPUT_CSV)
    
    # Determine x-axis (prefer snapshot_kimg, fallback to snapshot)
    if 'snapshot_kimg' in df.columns:
        x_col = 'snapshot_kimg'
        x_label = 'Training Progress (kimg)'
    elif 'snapshot' in df.columns:
        x_col = 'snapshot'
        x_label = 'Snapshot Number'
    else:
        x_col = None
        x_label = 'Index'
    
    # Find metric columns (exclude metadata)
    metric_cols = [col for col in df.columns if col not in ['snapshot', 'snapshot_kimg', 'timestamp', 'dataset_name']]
    
    print(f"\nMetric columns found: {metric_cols}")
    
    # Create plots
    fig, axes = plt.subplots(len(metric_cols), 1, figsize=(12, 4*len(metric_cols)))
    
    if len(metric_cols) == 1:
        axes = [axes]
    
    for ax, col in zip(axes, metric_cols):
        try:
            # Plot with both line and markers
            if x_col:
                ax.plot(df[x_col], df[col], marker='o', linewidth=2, markersize=6)
                ax.set_xlabel(x_label, fontsize=11)
            else:
                ax.plot(df[col], marker='o', linewidth=2, markersize=6)
                ax.set_xlabel('Snapshot Index', fontsize=11)
            
            ax.set_ylabel(col, fontsize=11)
            ax.set_title(f'{col} vs Training Progress', fontsize=12, fontweight='bold')
            ax.grid(True, alpha=0.3)
        except:
            ax.text(0.5, 0.5, f"Could not plot {col}\n(possibly non-numeric)", 
                   ha='center', va='center', transform=ax.transAxes)
    
    plt.tight_layout()
    plot_path = OUTPUT_CSV.replace('.csv', '_plots.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"\n✓ Plots saved to: {plot_path}")
    plt.show()

else:
    print("\n✗ No data to visualize.")

KeyboardInterrupt: 

## Cell 9: Summary Report

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

if len(all_metrics) > 0:
    print(f"\n📊 Results Summary:")
    print(f"  - Total snapshots evaluated: {len(pkl_files_to_test)}")
    print(f"  - Metrics successfully calculated: {len(all_metrics)}")
    print(f"  - Success rate: {100*len(all_metrics)/len(pkl_files_to_test):.1f}%")
    
    print(f"\n📁 Output Files:")
    print(f"  1. CSV (for Excel/analysis):")
    print(f"     {OUTPUT_CSV}")
    print(f"  2. JSON (for backup/programming):")
    print(f"     {OUTPUT_CSV.replace('.csv', '.json')}")
    try:
        print(f"  3. Excel (for easy viewing):")
        print(f"     {OUTPUT_CSV.replace('.csv', '.xlsx')}")
    except:
        pass
    print(f"  4. Plots (visualization):")
    print(f"     {OUTPUT_CSV.replace('.csv', '_plots.png')}")
    
    # Load and show final statistics
    df = pd.read_csv(OUTPUT_CSV)
    print(f"\n📈 Metric Statistics:")
    numeric_df = df.select_dtypes(include=[np.number])
    for col in numeric_df.columns:
        print(f"\n  {col}:")
        print(f"    Min: {numeric_df[col].min():.4f}")
        print(f"    Max: {numeric_df[col].max():.4f}")
        print(f"    Mean: {numeric_df[col].mean():.4f}")
        print(f"    Std: {numeric_df[col].std():.4f}")
    
    print(f"\n✓ All results saved successfully!")
    
else:
    print("\n✗ No metrics were generated. Please check the evaluation output above.")

print("\n" + "="*70)

## Notes & Troubleshooting

### If evaluation fails:

**CUDA Plugin Error** ("Could not find MSVC/GCC/CLANG")
- This is expected on Windows without Visual Studio. The code handles it.
- If still failing: Set `env["TORCH_COMPILE_DISABLE"] = "1"` in Cell 5

**Out of Memory Error**
- Reduce batch size: Modify `PYTHON_EXEC` to add `--batch 4` to cmd
- Use fewer snapshots: Increase `STEP_SIZE` to 10 or 20

**No PKL files found**
- Double-check `RUN_DIR` path
- Make sure training completed and saved snapshots

### Metric Meanings:
- **FID**: Fréchet Inception Distance (lower is better, measures quality/diversity)
- **IS**: Inception Score (higher is better, measures quality)
- **KID**: Kernel Inception Distance (lower is better, more stable than FID for small datasets)
- **LPIPS**: Learned Perceptual Image Patch Similarity (lower is better)

### For faster iteration:
- Use `STEP_SIZE = 10` for 100+ snapshots (evaluates every 10th)
- Use `METRICS = "kid50k_full"` instead of FID for more stable results on 100-shot
